In [1]:
from shared.main import * 

def load_embeddings(
    sub_id,
    *,
    neutrals=True,
    context="in-context",  # {"in-context", "no-context"}
    llm="openai",
    on_missing="none",     # {"none","empty","raise"}
    data_root=None,
):
    """
    Load decision embeddings for a subject.

    This version is safe to call from any folder because it does NOT use
    relative paths like ../data. It uses, in order:

      1. data_root argument, if provided
      2. global data_dir, if defined
      3. global PROJECT_ROOT / "data", if defined

    Directory selection, based on sub_id AFTER stripping 'sub-':
      - If int in [1, 100]:  data/other-samples/tavares/narratives/narrative-embeddings
      - If non-int:         data/other-samples/online/narratives/narrative-embeddings
      - Otherwise:          data/narratives/narrative-embeddings

    File naming:
      sub-<id>_decisions-in-context_<llm>.npz
      sub-<id>_decisions-no-context_<llm>.npz

    Neutral handling:
      - If neutrals=False, applies drop_neutral_trials() to the loaded embedding array.

    Missing-file behavior:
      - on_missing="none"  -> return None
      - on_missing="empty" -> return np.empty((0, 0), float)
      - on_missing="raise" -> raise FileNotFoundError
    """

    if on_missing not in {"none", "empty", "raise"}:
        raise ValueError("on_missing must be one of {'none', 'empty', 'raise'}")

    if context not in {"in-context", "no-context"}:
        raise ValueError("context must be 'in-context' or 'no-context'")

    # --------------------------------------------------
    # Resolve project data directory robustly
    # --------------------------------------------------

    if data_root is not None:
        data_base = Path(data_root).expanduser().resolve()

    elif "data_dir" in globals():
        data_base = Path(globals()["data_dir"]).expanduser().resolve()

    elif "PROJECT_ROOT" in globals():
        data_base = Path(globals()["PROJECT_ROOT"]).expanduser().resolve() / "data"

    else:
        raise RuntimeError(
            "Could not resolve data directory. Define `data_dir`, define `PROJECT_ROOT`, "
            "or pass `data_root='/path/to/project/data'`."
        )

    # --------------------------------------------------
    # Subject ID helpers
    # --------------------------------------------------

    def _strip_prefix(x) -> str:
        s = str(x).strip()
        return s[4:] if s.startswith("sub-") else s

    def _parse_int(s: str):
        s = str(s).strip()
        return int(s) if s.isdigit() else None

    def _format_sid(x) -> str:
        """
        Returns normalized subject id string WITH 'sub-' prefix.
        """
        raw = _strip_prefix(x)
        iv = _parse_int(raw)

        if iv is not None:
            core = f"{iv:02d}" if 1 <= iv <= 9 else str(iv)
        else:
            core = raw

        return f"sub-{core}"

    sid = _format_sid(sub_id)

    raw = _strip_prefix(sub_id)
    iv = _parse_int(raw)

    # --------------------------------------------------
    # Select embeddings directory
    # --------------------------------------------------

    if iv is not None and 1 <= iv <= 100:
        emb_dir = (
            data_base
            / "other-samples"
            / "tavares"
            / "narratives"
            / "narrative-embeddings"
        )

    elif iv is None:
        emb_dir = (
            data_base
            / "other-samples"
            / "online"
            / "narratives"
            / "narrative-embeddings"
        )

    else:
        emb_dir = data_base / "narratives" / "narrative-embeddings"

    # --------------------------------------------------
    # File name
    # --------------------------------------------------

    ctxt_suffix = (
        "decisions-in-context"
        if context == "in-context"
        else "decisions-no-context"
    )

    fpath = emb_dir / f"{sid}_{ctxt_suffix}_{llm}.npz"

    # --------------------------------------------------
    # Missing behavior
    # --------------------------------------------------

    if not fpath.exists():
        if on_missing == "raise":
            raise FileNotFoundError(f"Embeddings not found: {fpath}")
        if on_missing == "empty":
            return np.empty((0, 0), dtype=float)
        return None

    # --------------------------------------------------
    # Load
    # --------------------------------------------------

    with np.load(fpath) as z:
        if "embedding" not in z.files:
            raise KeyError(
                f"Expected key 'embedding' in {fpath}. "
                f"Available keys: {z.files}"
            )

        arr = z["embedding"].astype(float)

    # --------------------------------------------------
    # Optionally drop neutral trials
    # --------------------------------------------------

    if not neutrals:
        arr = drop_neutral_trials(arr)

    return arr

def load_behavior(
    sub_id,
    neutrals=True,
    *,
    on_missing="none",
    data_root=None,
):
    """
    Load behavior dataframe for a given subject.

    This version is safe to call from any folder because it does NOT use
    relative paths like ../../../data. It uses, in order:

      1. data_root argument, if provided
      2. global data_dir, if defined
      3. global PROJECT_ROOT / "data", if defined

    Directory selection, based on sub_id AFTER stripping 'sub-':
      - If int in [1, 100]:  data/other-samples/tavares/preprocessed/behavior
      - If non-int:         data/other-samples/online/preprocessed/behavior
      - Otherwise:          data/preprocessed/behavior

    Subject ID normalization:
      - Always uses "sub-" prefix
      - If integer in [1, 9], zero-pads to 2 digits: 1 -> sub-01
      - Otherwise keeps original digits: 18001 -> sub-18001

    Missing-file behavior:
      - on_missing="none"  -> return None
      - on_missing="empty" -> return empty DataFrame
      - on_missing="raise" -> raise FileNotFoundError
    """

    if on_missing not in {"none", "empty", "raise"}:
        raise ValueError("on_missing must be one of {'none', 'empty', 'raise'}")

    # --------------------------------------------------
    # Resolve project data directory robustly
    # --------------------------------------------------

    if data_root is not None:
        data_base = Path(data_root).expanduser().resolve()

    elif "data_dir" in globals():
        data_base = Path(globals()["data_dir"]).expanduser().resolve()

    elif "PROJECT_ROOT" in globals():
        data_base = Path(globals()["PROJECT_ROOT"]).expanduser().resolve() / "data"

    else:
        raise RuntimeError(
            "Could not resolve data directory. Define `data_dir`, define `PROJECT_ROOT`, "
            "or pass `data_root='/path/to/project/data'`."
        )

    # --------------------------------------------------
    # Subject ID helpers
    # --------------------------------------------------

    def _strip_prefix(x) -> str:
        s = str(x).strip()
        return s[4:] if s.startswith("sub-") else s

    def _parse_int(s: str):
        s = str(s).strip()
        return int(s) if s.isdigit() else None

    def _format_sid(x) -> str:
        """
        Returns normalized subject id string WITH 'sub-' prefix.
        """
        raw = _strip_prefix(x)
        iv = _parse_int(raw)

        if iv is not None:
            core = f"{iv:02d}" if 1 <= iv <= 9 else str(iv)
        else:
            core = raw

        return f"sub-{core}"

    sid = _format_sid(sub_id)

    raw = _strip_prefix(sub_id)
    iv = _parse_int(raw)

    # --------------------------------------------------
    # Select behavior directory
    # --------------------------------------------------

    if iv is not None and 1 <= iv <= 100:
        beh_dir = data_base / "other-samples" / "tavares" / "preprocessed" / "behavior"

    elif iv is None:
        beh_dir = data_base / "other-samples" / "online" / "preprocessed" / "behavior"

    else:
        beh_dir = data_base / "preprocessed" / "behavior"

    fpath = beh_dir / f"{sid}.xlsx"

    # --------------------------------------------------
    # Load
    # --------------------------------------------------

    if not fpath.exists():
        if on_missing == "raise":
            raise FileNotFoundError(f"Behavior file not found: {fpath}")
        if on_missing == "empty":
            return pd.DataFrame()
        return None

    df = pd.read_excel(fpath)

    # --------------------------------------------------
    # Optionally drop neutral trials
    # --------------------------------------------------

    if not neutrals:
        if "character_role_num" in df.columns:
            role_col = "character_role_num"
        elif "char_role_num" in df.columns:
            role_col = "char_role_num"
        else:
            if on_missing == "raise":
                raise ValueError(
                    f"Behavior file has no role column: {fpath}. "
                    "Expected `character_role_num` or `char_role_num`."
                )
            if on_missing == "empty":
                return pd.DataFrame()
            return None

        df = df[df[role_col] != 6].reset_index(drop=True)

    return df

def load_subject_data(
    sub_id,
    *,
    atlas=None,
    glm_dir=None,
    load_sem=True,
    load_fmri=True,
    neutrals=True,
    llm="openai",
    emb_on_missing="none",
    beh_on_missing="none",
    do_dots_mapping=True,
    skip_sem_if_missing=True,
    verbose=True,
    sub_id_width=2,
):
    """
    Load one subject's behavior, optional embeddings, and optional atlas ROI fMRI betas.

    Important fMRI behavior:
      - ROI beta matrices are NOT voxel-filtered.
      - No finite-voxel filtering.
      - No zero-variance voxel filtering.
      - Each ROI keeps every voxel assigned to that ROI after atlas resampling.
    """

    def log(msg):
        if verbose:
            print(f"[load_subject_data sub={sub_id}] {msg}")

    def normalize_sub_id(sub_id):
        sid_raw = str(sub_id).strip()
        if sid_raw.startswith("sub-"):
            sid_raw = sid_raw[4:]

        is_numeric = sid_raw.isdigit()
        sid_int = int(sid_raw) if is_numeric else None
        sid_str = sid_raw.zfill(sub_id_width) if is_numeric else sid_raw
        sid_with_prefix = f"sub-{sid_str}"
        is_main_sample = bool(is_numeric and sid_int >= 100)

        return sid_raw, sid_str, sid_with_prefix, is_numeric, is_main_sample

    def get_role_col(behav):
        if "character_role_num" in behav.columns:
            return "character_role_num"
        if "char_role_num" in behav.columns:
            return "char_role_num"
        return None

    def find_subject_row(df, sid_raw, sid_str, sid_with_prefix, is_numeric):
        sid_col = df["sub_id"].astype(str)

        row = df[sid_col == str(sid_str)]
        if row.empty and is_numeric:
            row = df[sid_col == str(sid_raw)]
        if row.empty:
            row = df[sid_col == sid_with_prefix]

        return row

    def parse_atlas(atlas):
        import nibabel as nib

        if isinstance(atlas, str):
            atlas = pd.read_pickle(atlas) if atlas.endswith(".pkl") else nib.load(atlas)

        if isinstance(atlas, dict):
            atlas_img = atlas["image"]
            rois = atlas["rois"]

            if isinstance(rois, dict):
                atlas_codes = [int(k) for k in rois.keys()]
                atlas_labels = [str(v) for v in rois.values()]
            else:
                atlas_labels = list(rois)
                atlas_codes = list(range(1, len(atlas_labels) + 1))

            return atlas_img, atlas_codes, atlas_labels

        if isinstance(atlas, nib.Nifti1Image):
            atlas_img = atlas
            labels = np.unique(atlas_img.get_fdata().astype(int))
            labels = labels[labels > 0]
            atlas_codes = [int(i) for i in labels]
            atlas_labels = [f"ROI-{i}" for i in labels]

            return atlas_img, atlas_codes, atlas_labels

        raise ValueError(f"atlas must be dict, str, or Nifti1Image; got {type(atlas)}")

    def find_beta_path(glm_dir, sid_with_prefix):
        beta_candidates = [
            os.path.join(glm_dir, f"{sid_with_prefix}_decision_trials_beta.nii.gz"),
            os.path.join(glm_dir, sid_with_prefix, "beta_decisions.nii.gz"),
            os.path.join(glm_dir, sid_with_prefix, "beta_decisions_resampled.nii.gz"),
        ]

        beta_path = next((p for p in beta_candidates if os.path.exists(p)), None)

        if beta_path is None:
            msg = "Beta image not found. Checked:\n" + "\n".join(
                f"  - {p}" for p in beta_candidates
            )
            raise FileNotFoundError(msg)

        return beta_path

    def load_roi_betas(atlas, glm_dir, sid_with_prefix, T_beh):
        import nibabel as nib
        from nilearn import image

        atlas_img, atlas_codes, atlas_labels = parse_atlas(atlas)
        beta_path = find_beta_path(glm_dir, sid_with_prefix)

        log(f"Loading beta image: {beta_path}")

        beta_img = nib.load(beta_path)
        beta = beta_img.get_fdata().astype(float)

        if beta.ndim != 4:
            raise ValueError(f"Expected 4D beta image, got shape {beta.shape}")

        T_fmri = beta.shape[3]

        if T_beh > 0 and T_fmri != T_beh:
            raise ValueError(
                f"fMRI/behavior length mismatch for {sid_str}: "
                f"fmri={T_fmri}, behavior={T_beh}"
            )

        beta_flat = beta.reshape(-1, T_fmri)

        atlas_resamp = image.resample_to_img(
            atlas_img,
            beta_img,
            interpolation="nearest",
        )
        atlas_flat = atlas_resamp.get_fdata().astype(int).ravel()

        if atlas_flat.shape[0] != beta_flat.shape[0]:
            raise ValueError(
                f"Atlas/beta voxel mismatch after resampling for {sid_str}: "
                f"atlas voxels={atlas_flat.shape[0]}, beta voxels={beta_flat.shape[0]}"
            )

        roi_betas = {
            name: beta_flat[atlas_flat == int(code)].T
            for code, name in zip(atlas_codes, atlas_labels)
        }

        return beta_path, roi_betas

    # -------------------------
    # Normalize subject id
    # -------------------------

    sid_raw, sid_str, sid_with_prefix, is_numeric, is_main_sample = normalize_sub_id(sub_id)

    out = {"sub_id": sid_str}
    log(f"sid_str={sid_str}, main_sample={is_main_sample}")

    # -------------------------
    # Behavior
    # -------------------------

    try:
        behav = load_behavior(sid_str, neutrals=neutrals, on_missing=beh_on_missing)
    except Exception as e:
        raise RuntimeError(f"Behavior loading raised for sub {sid_str}") from e

    if behav is None:
        log("Behavior is None -> returning None")
        return None

    out["behavior"] = behav
    T_beh = len(behav)

    role_col = get_role_col(behav)
    if role_col is None and T_beh > 0:
        raise ValueError(
            f"Missing role column for sub {sid_str} "
            "(expected character_role_num or char_role_num)"
        )

    out["char_roles"] = behav[role_col].to_numpy(int) if role_col else np.array([], int)

    # -------------------------
    # Dots
    # -------------------------

    out.update({
        "dots": None,
        "dots_roles": None,
        "dots_affine": None,
    })

    dots_df = globals().get("data" if is_main_sample else "data_online", None)

    if isinstance(dots_df, pd.DataFrame) and "sub_id" in dots_df.columns:
        row = find_subject_row(dots_df, sid_raw, sid_str, sid_with_prefix, is_numeric)

        if not row.empty:
            try:
                dots_arr = get_coords(row.iloc[0], which="dots", include_neutral=True)
                dots_by_char = np.asarray(dots_arr)[0]

                roles = ["first", "second", "assistant", "powerful", "boss", "neutral"]

                try:
                    global_roles = list(CHARACTERS)
                    if set(global_roles) == set(roles) and global_roles != roles:
                        idx = [roles.index(r) for r in global_roles]
                        dots_by_char = dots_by_char[idx]
                        roles = global_roles
                except Exception:
                    pass

                out["dots"] = dots_by_char
                out["dots_roles"] = roles

                if (
                    do_dots_mapping
                    and T_beh > 0
                    and {"affil_coord", "power_coord"} <= set(behav.columns)
                ):
                    beh_xy = behav[["affil_coord", "power_coord"]].to_numpy(float)
                    beh_dots, affine = transform_beh_to_dots(
                        beh_xy,
                        dots_by_char,
                        anchor="end",
                    )
                    behav[["affil_coord_in_dots", "power_coord_in_dots"]] = beh_dots
                    out["dots_affine"] = affine

            except Exception as e:
                log(f"Dots attach/mapping skipped: {e}")
    else:
        log("No dots df found: missing `data`/`data_online` or not a DataFrame -> skipping dots.")

    # -------------------------
    # Embeddings
    # -------------------------

    out.update({
        "embds_ctxt": None,
        "embds_noctxt": None,
        "has_sem": False,
    })

    if load_sem:
        emb_ctxt = None
        emb_noctxt = None

        try:
            emb_ctxt = load_embeddings(
                sid_str,
                neutrals=neutrals,
                context="in-context",
                llm=llm,
                on_missing=emb_on_missing,
            )
        except Exception as e:
            log(f"Embeddings (in-context) skipped: {e}")

        try:
            emb_noctxt = load_embeddings(
                sid_str,
                neutrals=neutrals,
                context="no-context",
                llm=llm,
                on_missing=emb_on_missing,
            )
        except Exception as e:
            log(f"Embeddings (no-context) skipped: {e}")

        if emb_ctxt is not None and emb_noctxt is not None:
            out.update({
                "embds_ctxt": emb_ctxt,
                "embds_noctxt": emb_noctxt,
                "has_sem": True,
            })

            if T_beh > 0 and emb_ctxt.shape[0] != T_beh:
                raise ValueError(
                    f"Embedding/behavior length mismatch for {sid_str}: "
                    f"in-context={emb_ctxt.shape[0]}, behavior={T_beh}"
                )

            if T_beh > 0 and emb_noctxt.shape[0] != T_beh:
                raise ValueError(
                    f"Embedding/behavior length mismatch for {sid_str}: "
                    f"no-context={emb_noctxt.shape[0]}, behavior={T_beh}"
                )

        elif not skip_sem_if_missing:
            log("Semantic embeddings missing and skip_sem_if_missing=False -> returning None")
            return None

    # -------------------------
    # fMRI
    # -------------------------

    load_fmri = bool(load_fmri and atlas is not None and glm_dir is not None)

    if load_fmri:
        beta_path, roi_betas = load_roi_betas(atlas, glm_dir, sid_with_prefix, T_beh)
        out["beta_path"] = beta_path
        out["roi_betas"] = roi_betas
        out["has_fmri"] = True
    else:
        out["roi_betas"] = None
        out["has_fmri"] = False

    log("Success.")
    return out

# atlas_pkl = PROJECT_ROOT / "masks" / "atlases" / "Tavares2015_spheres_atlas.pkl"
# out_pkl   = PROJECT_ROOT / "analyses" / "lss_decision" / "subject_data_Tavares.pkl"
# glm_dir   = PROJECT_ROOT / "analyses" / "lss_decision" / "glms"

# all_incl_subs = incl_subs
# subject_data = {}
# for sub_id in tqdm(all_incl_subs):
#     try:
#         sd = load_subject_data(
#             sub_id,
#             atlas=str(atlas_pkl),
#             glm_dir=str(glm_dir),
#             load_sem=True,
#             load_fmri=True,
#             verbose=True
#         )
#         subject_data[sub_id] = sd
#     except FileNotFoundError as e:
#         print(f"[Missing files] {sub_id}: {e}")
#     except Exception as e:
#         # keep the original behavior but make it clear which subject failed
#         print(f"[Error] {sub_id}: {e}")

# pickle_file(subject_data, str(out_pkl))

In [ ]:
from transformers import pipeline
sentiment_model = pipeline("sentiment-analysis",  model="cardiffnlp/twitter-roberta-base-sentiment",  return_all_scores=True) 

def calculate_compound_score(sentiment):
    
    positive = sentiment[[k for k in sentiment.keys() if 'positivity' in k][0]]
    negative = sentiment[[k for k in sentiment.keys() if 'negativity' in k][0]]
    neutral  = sentiment[[k for k in sentiment.keys() if 'neutrality' in k][0]]

    # Normalize scores --> probabilities
    total = positive + negative + neutral
    positive_norm = positive / total
    negative_norm = negative / total
    neutral_norm  = neutral / total
    
    # Simple compound score calculation
    # - scale by neutral to reduce the compound score when the sentiment is mostly neutral
    compound_score = (positive_norm - negative_norm) * (1 - neutral_norm)
    return np.round(compound_score, 3)

def run_sentiment_analysis(text):
    sentiment = sentiment_model(text)[0]
    sentiment = {sent['label']: np.round(sent['score'], 3) for sent in sentiment}
    label_mapping = {'LABEL_0': 'negativity', 'LABEL_1': 'neutrality', 'LABEL_2': 'positivity'} 
    sentiment = {label_mapping[k]: v for k, v in sentiment.items() if k in label_mapping}
    sentiment['compound'] = calculate_compound_score(sentiment)
    return sentiment
